In [ ]:
import numpy as np
import optuna
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

from features import feature_engeneering, get_fe_parametrs
from folds import FOLDS, X, y


def objective_ocsvm(trial):

    params = {
        'kernel': 'rbf',
        'nu': trial.suggest_float('nu', 0.01, 0.15),
        'gamma': trial.suggest_float('gamma', 1e-4, 1e1, log=True),
    }

    aucs = []

    for fold, (tr_idx, val_idx) in enumerate(FOLDS):
        X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

        qt, train_median, train_mad, tsh_q, fti_q, tt4_q = get_fe_parametrs(X_tr)

        X_tr_f = feature_engeneering(df=X_tr, qt=qt, train_median=train_median,
                                     train_mad=train_mad, tsh_quantile=tsh_q,
                                     fti_quantile=fti_q, tt4_quantile=tt4_q)
        X_val_f = feature_engeneering(df=X_val, qt=qt, train_median=train_median,
                                      train_mad=train_mad, tsh_quantile=tsh_q,
                                      fti_quantile=fti_q, tt4_quantile=tt4_q)

        X_tr_normal = X_tr_f[y_tr == 0]
        
        scaler = StandardScaler()
        scaler.fit(X_tr_normal)
        X_tr_scaled = scaler.transform(X_tr_normal)
        X_val_scaled = scaler.transform(X_val_f)

        model = OneClassSVM(**params)
        model.fit(X_tr_scaled)

        val_scores = -model.decision_function(X_val_scaled)
        pr_auc = average_precision_score(y_val, val_scores)
        aucs.append(pr_auc)

        trial.report(float(np.mean(aucs)), fold)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(aucs))

study_ocsvm = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=131),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1)
)

study_ocsvm.optimize(objective_ocsvm, n_trials=100)
print(f'Best params: {study_ocsvm.best_params}')
print(f'Best value: {study_ocsvm.best_value}')

c:\Users\DNS\PycharmProjects\ds&ml\anomaly_and_pca\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-11-22 16:41:01,503] A new study created in memory with name: no-name-4a6f950e-28c6-4395-990b-675f85a15cad
[I 2026-11-22 16:41:02,946] Trial 0 finished with value: 0.349883878930513 and parameters: {'nu': 0.10102150476604485, 'gamma': 5.502319810268872}. Best is trial 0 with value: 0.349883878930513.
[I 2026-11-22 16:41:03,469] Trial 1 finished with value: 0.530392623220725 and parameters: {'nu': 0.06432404414976087, 'gamma': 0.1608721220999993}. Best is trial 1 with value: 0.530392623220725.
[I 2026-11-22 16:41:03,766] Trial 2 finished with value: 0.4256466605516337 and parameters: {'nu': 0.10667168761690309, 'gamma': 0.0004295824582316458}. Best is trial 1 with value: 0.530392623220725.
[I 2026-11-22 16

Best params: {'nu': 0.13016916406071138, 'gamma': 0.25885189066907827}
Best value: 0.5405396782300167


In [ ]:
best_params = study_ocsvm.best_params
oof_ocsvm = np.zeros(len(X))
synth_ocsvm = []

for fold, (tr_idx, val_idx) in enumerate(FOLDS):
    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    qt, train_median, train_mad, tsh_q, fti_q, tt4_q = get_fe_parametrs(X_tr)

    X_tr_f = feature_engeneering(
        df=X_tr, qt=qt, train_median=train_median,
        train_mad=train_mad, tsh_quantile=tsh_q, 
        fti_quantile=fti_q, tt4_quantile=tt4_q 
    )
    X_val_f = feature_engeneering(
        df=X_val, qt=qt, train_median=train_median,
        train_mad=train_mad, tsh_quantile=tsh_q,
        fti_quantile=fti_q, tt4_quantile=tt4_q
    )

    X_tr_normal = X_tr_f[y_tr == 0]

    scaler = StandardScaler()
    scaler.fit(X_tr_normal)

    X_tr_scaled = scaler.transform(X_tr_normal)
    X_val_scaled = scaler.transform(X_val_f)

    X_synth = X_tr_normal.copy()
    for col in X_tr_normal.columns:
        X_synth[col] = np.random.permutation(X_synth[col].values)
    X_synth_scaled = scaler.transform(X_synth)


    model = OneClassSVM(**best_params)
    model.fit(X_tr_scaled)

    oof_ocsvm[val_idx] = -model.decision_function(X_val_scaled)
    synth_ocsvm.extend(-model.decision_function(X_synth_scaled))

np.save('oof_ocsvm.npy', oof_ocsvm)
np.save('synth_ocsvm.npy', synth_ocsvm)

: 